# Fase 1. Exploración de datos MiBici

Este notebook usa el paquete `mibici` ya refactorizado. La lógica de limpieza, features y gráficas vive en los módulos; aquí solo se revisan los resultados.

In [1]:
import pandas as pd
from mibici.config import CleaningConfig
from mibici.dataset import MiBiciLoader
from mibici.cleaning import TripCleaner
from mibici.plots import EDAReport

## Carga de datos crudos

In [2]:
loader = MiBiciLoader()
stations = loader.load_stations()
trips = loader.load_trips()
trips.info()

2026-10-25 11:46:58.826 | INFO     | mibici.dataset:load_trips:43 - Leyendo datos_abiertos_2026_07.csv


2026-10-25 11:46:59.602 | INFO     | mibici.dataset:load_trips:43 - Leyendo datos_abiertos_2026_08.csv


2026-10-25 11:47:00.329 | INFO     | mibici.dataset:load_trips:43 - Leyendo datos_abiertos_2026_09.csv


2026-10-25 11:47:01.442 | INFO     | mibici.dataset:load_trips:49 - Viajes cargados: 1,271,837


<class 'pandas.DataFrame'>
RangeIndex: 1271837 entries, 0 to 1271836
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   trip_id         1271837 non-null  int64  
 1   user_id         1271837 non-null  int64  
 2   gender          1263559 non-null  string 
 3   birth_year      1271716 non-null  float64
 4   start_time      1271837 non-null  str    
 5   end_time        1271837 non-null  str    
 6   origin_id       1271837 non-null  int64  
 7   destination_id  1271837 non-null  int64  
 8   source_month    1271837 non-null  str    
dtypes: float64(1), int64(4), str(3), string(1)
memory usage: 143.4 MB


In [3]:
trips.isna().sum()

trip_id              0
user_id              0
gender            8278
birth_year         121
start_time           0
end_time             0
origin_id            0
destination_id       0
source_month         0
dtype: int64

In [4]:
trips['birth_year'].describe()

count    1.271716e+06
mean     1.990824e+03
std      1.491056e+01
min      1.990000e+02
25%      1.985000e+03
50%      1.994000e+03
75%      1.999000e+03
max      2.010000e+03
Name: birth_year, dtype: float64

## Limpieza

Cada regla se registra con el número de filas que elimina.

In [5]:
cleaner = TripCleaner(stations, CleaningConfig())
clean = cleaner.clean(trips)
cleaner.report_frame()

2026-10-25 11:47:04.862 | INFO     | mibici.cleaning:_log:28 - duplicados_exactos: -758 filas


2026-10-25 11:47:05.374 | INFO     | mibici.cleaning:_log:28 - fechas_invalidas_o_negativas: -0 filas


2026-10-25 11:47:05.449 | INFO     | mibici.cleaning:_log:28 - duracion_menor_1_min: -52,779 filas


2026-10-25 11:47:05.549 | INFO     | mibici.cleaning:_log:28 - misma_estacion_menor_2_min: -8,871 filas


2026-10-25 11:47:05.611 | INFO     | mibici.cleaning:_log:28 - duracion_mayor_180_min: -151 filas


2026-10-25 11:47:05.684 | INFO     | mibici.cleaning:_log:28 - estacion_fuera_de_catalogo: -1,937 filas


,regla,filas_antes,filas_eliminadas,filas_despues,nota
0,duplicados_exactos,1271837,758,1271079,
1,fechas_invalidas_o_negativas,1271079,0,1271079,
2,duracion_menor_1_min,1271079,52779,1218300,anclaje fallido o arrepentimiento
3,misma_estacion_menor_2_min,1218300,8871,1209429,bicicleta devuelta de inmediato
4,duracion_mayor_180_min,1209429,151,1209278,"outlier, bicicleta no anclada"
5,estacion_fuera_de_catalogo,1209278,1937,1207341,
6,anio_nacimiento_invalido_a_nulo,1207341,0,1207341,89 valores fuera de rango convertidos a nulo


## Estadísticas descriptivas

In [6]:
eda = EDAReport(clean, stations)
pd.Series(eda.summary()).to_frame('valor')

,valor
viajes,1207341
usuarios_unicos,22514
estaciones_origen,469
viajes_dia_promedio,13123.271739
viajes_dia_habil,14268.242424
viajes_dia_fds,10216.807692
dia_max,2026-09-30
viajes_dia_max,20686
dia_min,2026-07-25
viajes_dia_min,8052


In [7]:
eda.duration_table()

,duracion_min
count,1207341.00
mean,10.87
std,7.34
min,1.00
25%,5.80
50%,9.18
75%,14.05
95%,24.33
99%,33.75
max,179.82


In [8]:
clean.groupby('gender', observed=True)['duration_min'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
gender,,,,,,,,
F,327364.0,11.28,7.48,1.00,6.13,9.58,14.53,178.57
M,872127.0,10.71,7.28,1.00,5.68,9.03,13.87,179.82
NE,7850.0,11.26,7.92,1.05,6.07,9.53,14.43,174.32
